# 4장 프롬프트 설계 (풀이)

검증용 풀이 노트북입니다. 공유하지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

import csv
import re

with open(f'{DATA}/문의.csv', encoding='utf-8') as file:
    inquiries = list(csv.DictReader(file))        # 문의 60건. 열: 번호, 문의, 라벨, 긴급, 비고
LABELS = ['설비 이상', '품질', '안전', '문서와 데이터 요청']


def numbered(items):
    """문의 목록을 "번호: 문의" 줄로 이어 붙입니다"""
    return '\n'.join(f"{item['번호']}: {item['문의']}" for item in items)


def parse_labels(text):
    """응답에서 "Q01: 라벨" 형식의 줄을 찾아 {번호: 라벨}로 돌려줍니다"""
    found = {}
    for line in text.splitlines():
        # "Q01: 라벨", "**Q01**: 라벨", "| Q01 | 라벨 |" 같은 줄을 모두 읽습니다
        match = re.match(r'[\s|*-]*(Q\d{2})[\s*]*[:：|][\s*]*(.+?)[\s*|]*$', line)
        if match:
            found[match.group(1)] = match.group(2).split('|')[0].strip()   # 표라면 첫 칸만
    return found


def accuracy(predicted, items):
    """정답 라벨과 같은 건의 비율"""
    return sum(predicted.get(item['번호']) == item['라벨'] for item in items) / len(items)


print(f'문의 {len(inquiries)}건, 라벨 {LABELS}')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 4-1. 모호한 지시와 구체적 지시

교재의 절: "프롬프트가 응답을 바꾸는 원리"

문의 10건을 "분류해 주세요"만으로 두 번, 라벨 목록과 "번호: 라벨" 형식을 정해 두 번 보냅니다. 응답의 길이, 읽어 낸 라벨의 종류, 응답 앞부분이 실행마다 어떻게 다른지 봅니다. 이 셀은 API를 4번 호출합니다.

In [ ]:
sample = [inquiries[i] for i in [0, 10, 20, 30, 40, 42, 44, 49, 11, 31]]   # 라벨이 섞이게 고른 10건
vague = '다음 문의들을 분류해 주세요.\n\n' + numbered(sample)
specific = ('다음 문의를 [설비 이상, 품질, 안전, 문서와 데이터 요청] 중 하나로 분류하세요.\n'
            '한 줄에 한 건씩 "번호: 라벨" 형식으로만 답하세요.\n\n' + numbered(sample))
for name, prompt in [('모호한 지시', vague), ('구체적 지시', specific)]:
    for attempt in range(1, 3):
        reply = llm.ask(prompt, model='소형', max_tokens=2000)
        labels = parse_labels(reply.text)                        # "Q01: 라벨" 줄만 읽습니다
        valid = sum(label in LABELS for label in labels.values())   # 네 라벨 가운데 하나로 읽힌 건수
        print(f'[{name} {attempt}회] 응답 {len(reply.text):,}자, 네 라벨로 읽힌 건 {valid}/10')
        print('   앞부분:', reply.text.strip()[:80].replace('\n', ' / '))

## 예제 4-2. few-shot의 효과와 치우침

교재의 절: "few-shot"

헷갈리거나 정보가 빠진 문의 16건(Q45~Q60)을 예시 없이, 라벨이 서로 다른 예시 3개로, 모두 "안전"인 예시 3개로 분류하고 정확도와 라벨별 건수를 비교합니다. 이 셀은 API를 3번 호출합니다.

In [ ]:
evaluation = inquiries[44:60]                    # Q45~Q60: 헷갈리는 11건과 정보가 빠진 5건
instruction = ('다음 문의를 [설비 이상, 품질, 안전, 문서와 데이터 요청] 중 하나로 분류하세요.\n'
               '한 줄에 한 건씩 "번호: 라벨" 형식으로만 답하세요.\n')


def with_examples(numbers):
    """예시로 쓸 문의(0부터 센 위치)를 "예시) 문의 → 라벨" 줄로 붙입니다"""
    return instruction + '\n'.join(f"예시) {inquiries[n]['문의']} → {inquiries[n]['라벨']}" for n in numbers) + '\n'


conditions = {'예시 없음': instruction,
              '예시 3개(라벨 세 종류)': with_examples([40, 41, 42]),     # Q41 설비 이상, Q42 문서와 데이터 요청, Q43 안전
              '예시 3개(모두 안전)': with_examples([20, 21, 22])}      # Q21~Q23 모두 안전
for name, prompt in conditions.items():
    predicted = parse_labels(llm.ask(prompt + '\n' + numbered(evaluation), model='소형', max_tokens=2000).text)
    counts = {label: list(predicted.values()).count(label) for label in LABELS}
    print(f'[{name}] 읽은 라벨 {len(predicted)}건, 정확도 {accuracy(predicted, evaluation):.0%}, 라벨별 건수 {counts}')

## 예제 4-3. 빈칸을 채우는 모델

교재의 절: "형식과 제약"

설비 번호, 담당자, 시각이 빠진 문의 5건(Q56~Q60)으로 "번호 | 설비 번호 | 담당자 | 발생 시각 | 요청 내용" 표를 요구합니다. 제약 없이 한 번, "없는 값은 확인 필요"라는 제약 한 줄을 더해 한 번 보내고, 문의에 없는 값이 표에 생기는지 봅니다. 이 셀은 API를 2번 호출합니다.

In [ ]:
missing = inquiries[55:60]                       # Q56~Q60: 설비 번호, 담당자, 시각이 빠진 문의
table = ('다음 문의마다 "번호 | 설비 번호 | 담당자 | 발생 시각 | 요청 내용" 표의 한 행을 만드세요.\n\n'
         + numbered(missing))
guarded = table.replace('만드세요.', '만드세요. 문의에 없는 값은 지어내지 말고 "확인 필요"로 적으세요.')
for name, prompt in [('제약 없음', table), ('제약 한 줄 추가', guarded)]:
    reply = llm.ask(prompt, model='소형', max_tokens=2000)
    print(f'[{name}]\n{reply.text.strip()}\n')

## 실습 4-1. 판정 기준으로 정확도 올리기

교재의 절: "개념 확인 실습"

헷갈리거나 정보가 빠진 문의 20건(Q41~Q60)을 분류하는 프롬프트에 판정 기준을 적어 정확도를 올립니다. 먼저 `my_guess`에 예상 정확도를 적고, 틀린 문의를 보고 기준을 한 번 고쳐 다시 실행합니다. 정확도가 예제 4-2의 "예시 없음"보다 올랐는지, 남은 오답이 어떤 문의인지 한 문장으로 설명하면 끝납니다. 이 셀은 실행할 때마다 API를 1번 호출합니다.

In [ ]:
hard = inquiries[40:60]                         # Q41~Q60: 헷갈리는 15건과 정보가 빠진 5건
my_prompt = ('다음 문의를 [설비 이상, 품질, 안전, 문서와 데이터 요청] 중 하나로 분류하세요.\n'
             '한 줄에 한 건씩 "번호: 라벨" 형식으로만 답하세요.\n'
             '판정 기준은 위에서부터 차례로 적용합니다.\n'
             '1. 사람이 다치거나 다칠 위험이 있으면 안전\n'
             '2. 문의자가 원하는 것이 문서나 기록의 전달이면 문서와 데이터 요청\n'
             '3. 설비의 고장, 알람, 이상 동작이거나 불량의 원인 설비가 특정되면 설비 이상\n'
             '4. 제품 결과나 품질 판정에 이상이 있고 원인 설비가 특정되지 않으면 품질\n')
my_guess = 0.8

reply = llm.ask(my_prompt + '\n' + numbered(hard), model='소형', max_tokens=2000)
predicted = parse_labels(reply.text)
print(f'읽은 라벨 {len(predicted)}건(종료 이유 {reply.stop_reason}), 정확도 {accuracy(predicted, hard):.0%} (예측 {my_guess})')
for item in hard:
    if predicted.get(item['번호']) != item['라벨']:
        print(f"틀림 {item['번호']}: {item['문의']} → {predicted.get(item['번호'])} (정답 {item['라벨']})")

## 실습 4-2. 체인으로 나누고 단계 사이에서 검사하기

교재의 절: "개념 확인 실습"

같은 20건에 대해 라벨과 긴급 여부를 한 번에 묻는 경우와, 라벨을 먼저 받고 검사한 뒤 긴급 여부를 따로 묻는 두 단계 체인을 비교합니다. 분류 기준은 실습 4-1에서 작성한 `my_prompt`를 그대로 씁니다. `invalid_labels()`는 허용 목록(`LABELS`)에 없는 라벨이 붙은 번호를 돌려주게, `step2_prompt()`는 문의와 1단계 라벨을 함께 주고 긴급 여부만 "번호: 예" 또는 "번호: 아니오"로 묻게 작성합니다. 긴급의 뜻(`URGENT`)은 두 방식에 똑같이 넣습니다. 어느 쪽의 긴급 판단이 정확할지 먼저 예측하고, 결과의 차이를 체인의 장단점으로 설명하면 끝납니다. 이 셀은 API를 3번 호출합니다.

In [ ]:
hard = inquiries[40:60]
URGENT = '긴급은 지금 조치하지 않으면 사람이 다치거나, 생산이 멈추거나, 불량이 출하될 수 있는 문의입니다.\n'


def invalid_labels(predicted):
    """허용 목록(LABELS)에 없는 라벨이 붙은 번호의 목록"""
    return [number for number, label in predicted.items() if label not in LABELS]


def step2_prompt(labels):
    """문의와 1단계 라벨을 함께 주고, 긴급 여부만 "번호: 예" 또는 "번호: 아니오"로 묻는 프롬프트"""
    lines = '\n'.join(f"{item['번호']}: {item['문의']} (라벨: {labels.get(item['번호'])})" for item in hard)
    return URGENT + '각 문의가 긴급인지 판단하세요. 한 줄에 한 건씩 "번호: 예" 또는 "번호: 아니오"로만 답하세요.\n\n' + lines


assert step2_prompt({'Q41': '설비 이상'}), 'step2_prompt()를 먼저 작성합니다'   # 빈칸이면 API를 부르기 전에 멈춥니다
# 한 번에: 라벨과 긴급 여부를 함께(분류 기준은 실습 4-1의 my_prompt)
once = llm.ask(my_prompt + URGENT + '"번호: 라벨 / 예 또는 아니오"로 답하세요.\n\n' + numbered(hard),
               model='소형', max_tokens=3000).text
# 체인 1단계: 라벨만, 단계 사이에서 검사
step1 = parse_labels(llm.ask(my_prompt + '\n' + numbered(hard), model='소형', max_tokens=2000).text)
print('1단계 검사에 걸린 번호:', invalid_labels(step1))
# 체인 2단계: 긴급 여부만
step2 = parse_labels(llm.ask(step2_prompt(step1), model='소형', max_tokens=2000).text)

pairs = {number: value.split('/') for number, value in parse_labels(once).items()}
once_labels = {number: parts[0].strip() for number, parts in pairs.items()}
once_urgent = {number: parts[-1].strip() for number, parts in pairs.items()}
urgent_score = lambda answers: sum(answers.get(item['번호']) == item['긴급'] for item in hard) / len(hard)
print('한 번에 응답 앞부분:', once.strip()[:60].replace('\n', ' / '))
print(f'한 번에: 라벨 {accuracy(once_labels, hard):.0%}, 긴급 {urgent_score(once_urgent):.0%}')
print(f'체인:   라벨 {accuracy(step1, hard):.0%}, 긴급 {urgent_score(step2):.0%}')